In [2]:
import polars as pl
import pandas as pd

## Polars

In [ ]:
burdens_dir = "/home/dnanexus/data_dir/burdens/"

# 1. Use scan_parquet for lazy loading (doesn't read data yet)
#    This saves memory by only reading what is needed during execution.
loftee = pl.scan_parquet(burdens_dir + "loftee_hc.parquet")
am = pl.scan_parquet(burdens_dir + "am_pathogenicity.parquet")

# 2. Concatenate and Sum
total_burden = (
    pl.concat(
        [loftee, am],
        how="diagonal"  # "diagonal" ensures columns (genes) are merged correctly even if they differ
    )
    .fill_null(0)       # Replace missing values (from the diagonal merge) with 0
    .group_by("sample") # Group by sample ID to merge the rows
    .sum()              # Sum the values for duplicate samples
    .sort("sample")     # Optional: Ensure output is sorted
    .collect(engine='streaming') # streaming=True processes in chunks to prevent crashes
)

total_burden

/tmp/ipykernel_77914/91393387.py:18: DeprecationWarning: the `streaming` parameter was deprecated in 1.25.0; use `engine` instead.
  .collect(streaming=True) # streaming=True processes in chunks to prevent crashes


## Pandas

In [ ]:
burdens_dir = "/home/dnanexus/data_dir/burdens/"

loftee = pd.read_parquet(burdens_dir + "loftee_hc.parquet")
am = pd.read_parquet(burdens_dir + "am_pathogenicity.parquet")

In [3]:
loftee.shape

(469835, 18054)

In [4]:
am.shape

(469835, 18054)

In [5]:
loftee = loftee.set_index("sample")
am = am.set_index("sample")

In [7]:
total_burden = loftee.add(am, fill_value=0)

total_burden = total_burden.reset_index().sort_values("sample")

total_burden

: 